# Stack validation on AirfRANS (Kaggle)

Trains a small Transolver on the AirfRANS `scarce` task on a Kaggle T4. Two
panels (surface Cp, |U|) on a held-out airfoil are saved as
`data/samples/airfrans_validation.png`. Acceptance bar is per-channel rel-L2
within ~2x the published Transolver number.

**Before Run All**: in Settings, attach `zeteixeira/airfrans-dataset` as
input, set Accelerator to GPU T4, Internet on.


In [ ]:
# clone or update the repo into /kaggle/working
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/zeteixeira03/transolver-hypersonic.git"
PROJECT_ROOT = Path("/kaggle/working/transolver-hypersonic")

if PROJECT_ROOT.exists():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=False)
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
!pip install --quiet 'airfrans>=0.1.5' 'einops>=0.7'

In [ ]:
# locate AirfRANS Dataset folder under attached input (any depth)
from pathlib import Path
hits = list(Path("/kaggle/input").rglob("manifest.json"))
assert hits, "AirfRANS dataset not attached. Add zeteixeira/airfrans-dataset as input."
DATA_ROOT = hits[0].parent
print("AirfRANS root:", DATA_ROOT)


In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


## Datasets and norm stats

In [ ]:
from src.data.airfrans import AirfRANSDataset, compute_norm_stats, load_split

TASK = "scarce"
SUBSAMPLE = 32_000

train_arrs, train_names = load_split(DATA_ROOT, task=TASK, train=True)
print(f"train: {len(train_arrs)} cases, first shape: {train_arrs[0].shape}")

stats = compute_norm_stats(train_arrs)
print("x_mean", stats.x_mean.numpy().round(3))
print("x_std ", stats.x_std.numpy().round(3))
print("y_mean", stats.y_mean.numpy().round(3))
print("y_std ", stats.y_std.numpy().round(3))

train_ds = AirfRANSDataset(train_arrs, train_names, stats, subsample=SUBSAMPLE)

test_arrs, test_names = load_split(DATA_ROOT, task=TASK, train=False)
print(f"test: {len(test_arrs)} cases")
test_ds = AirfRANSDataset(test_arrs, test_names, stats, subsample=None)


## Model, optimizer, scaler

In [ ]:
import random, numpy as np
torch.manual_seed(0); np.random.seed(0); random.seed(0)
from torch.utils.data import DataLoader
from src.models.transolver import Transolver
from src.training.loop import TrainConfig, collate_single, train_one_epoch, evaluate

model = Transolver(
    space_dim=7, fun_dim=0, out_dim=4,
    n_hidden=128, n_layers=4, n_head=8,
    mlp_ratio=2, slice_num=32, dropout=0.0,
    unified_pos=True, grid_ref=8, grid_bounds=(-2.0, 4.0, -1.5, 1.5),
).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

cfg = TrainConfig(
    epochs=200, lr=5e-4, weight_decay=0.0,
    surface_weight=1.0, batch_size=1, device=DEVICE,
    val_every=20, amp=False,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_single)


## Train

In [ ]:
import time
from tqdm.auto import tqdm

history = []
pbar = tqdm(range(cfg.epochs))
for epoch in pbar:
    t0 = time.time()
    train_metrics = train_one_epoch(model, loader, optimizer, cfg, scaler=scaler)
    scheduler.step()
    dt = time.time() - t0
    row = {"epoch": epoch, "sec": dt, **train_metrics}

    if epoch % cfg.val_every == cfg.val_every - 1 or epoch == cfg.epochs - 1:
        row.update(evaluate(model, test_ds, stats, DEVICE))

    history.append(row)
    pbar.set_postfix({k: f"{v:.3f}" for k, v in row.items() if isinstance(v, (int, float))})

    if epoch == 0:
        eta_min = dt * cfg.epochs / 60
        print(f"first epoch {dt:.1f}s -> projected ~{eta_min:.1f} min for {cfg.epochs} epochs")
        if eta_min > 480:
            print("WARNING: projected runtime exceeds a 9h Kaggle session. Consider reducing epochs.")

print("final:", history[-1])


## Acceptance figure and checkpoint

In [ ]:
from src.eval.plots import plot_acceptance

samples_dir = PROJECT_ROOT / "data" / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)
fig_path = samples_dir / "airfrans_validation.png"

fig, metrics = plot_acceptance(
    model=model, dataset=test_ds, stats=stats, device=DEVICE,
    sim_index=0, save_path=fig_path,
)
print("saved:", fig_path)
print("per-channel rel-L2:", metrics)


In [ ]:
ckpt_dir = PROJECT_ROOT / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
torch.save({
    "state_dict": model.state_dict(),
    "stats": {"x_mean": stats.x_mean, "x_std": stats.x_std,
              "y_mean": stats.y_mean, "y_std": stats.y_std},
    "history": history,
    "config": cfg.__dict__,
}, ckpt_dir / "airfrans_transolver_small.pt")
print("checkpoint saved")
